# 33. 하드 스위치 게이트

32번은 폴백을 가우시안 가중치로 섞었다.

`w = exp(-(|예측 − 중앙값| / tau)^2) × [ |폴백 − 중앙값| > THR ]`
`예측 = (1-w)·SVR + w·폴백`

볼록결합은 제곱오차를 줄일 때 최적이다. **대회 지표는 MAE 이고, MAE 의 최적 예측은
조건부 중앙값이다.** 이 문제의 사후분포는 이봉형이다 — 모델이 짝을 찾았으면 그 값 근처,
못 찾았으면 레벨 추정치 근처다. 두 봉우리 사이를 평균 내면 어느 쪽도 아닌 값이 나온다.

여기서는 섞지 않고 **둘 중 하나를 고른다.**

`예측 = 폴백  (|예측 − 중앙값| < c 이고 |폴백 − 중앙값| > THR 일 때)`
`예측 = SVR   (그 외)`

`tau` 와 `wmax` 가 사라지고 조건이 하나로 합쳐진다.

## 1. 설정

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer, RobustScaler
from sklearn.svm import SVR

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
SEEDS = [42, 2024, 7, 123, 999, 2025]
NEW_SEEDS = [31337, 5, 2718, 1618]

NUM = ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
       'diastolic_blood_pressure', 'glucose', 'bone_density']
CAT = ['gender', 'activity', 'smoke_status', 'medical_history',
       'family_medical_history', 'sleep_pattern', 'edu_level']

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
y = train['stress_score'].to_numpy(dtype=float)
print(train.shape, test.shape)

(3000, 18) (3000, 17)


## 2. 전처리와 모델

26·29·32번과 동일하다. `mean_working` 은 거리 계산에 넣지 않는다.

In [2]:
def numeric(df):
    x = df[NUM].copy()
    x['bmi'] = (df['weight'] / (df['height'] / 100) ** 2).round(2)
    return x.to_numpy(dtype=float)

scaler = RobustScaler().fit(numeric(train))
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False,
                    dtype=float).fit(train[CAT].fillna('Unknown'))

def build(df):
    return np.hstack([scaler.transform(numeric(df)),
                      ohe.transform(df[CAT].fillna('Unknown'))])

X, X_test = build(train), build(test)
LVL = train['mean_working'].round(0).fillna(-1).to_numpy(float)
LVL_TEST = test['mean_working'].round(0).fillna(-1).to_numpy(float)
MEDIAN = float(np.median(y))

def model():
    return TransformedTargetRegressor(
        regressor=SVR(C=4.0, gamma=2.0, kernel='rbf', epsilon=0.0),
        transformer=QuantileTransformer(output_distribution='normal',
                                        n_quantiles=1000, random_state=RANDOM_STATE))

def iso_table(lv_tr, y_tr, lv_target):
    m = lv_tr >= 0
    ir = IsotonicRegression(out_of_bounds='clip').fit(lv_tr[m], y_tr[m])
    return np.where(lv_target >= 0,
                    ir.predict(np.where(lv_target >= 0, lv_target, 0)),
                    y_tr.mean()).astype(float)

print(f'X {X.shape}, 타겟 중앙값 {MEDIAN}')

X (3000, 32), 타겟 중앙값 0.48


## 3. 두 규칙

32번에서 확인한 레벨별 정보량은 그대로 쓴다. `mean_working` 등위회귀 값이 중앙값과
얼마나 떨어져 있는지로 폴백을 쓸 레벨을 가른다.

In [3]:
lv_all = np.r_[np.arange(4., 17.), -1.]
iso_all = iso_table(LVL, y, lv_all)
info = pd.DataFrame({
    'n': [int((LVL == v).sum()) for v in lv_all],
    '등위회귀': iso_all.round(4),
    '|중앙값과 차이|': np.abs(iso_all - MEDIAN).round(4),
}, index=[int(v) if v > 0 else '결측' for v in lv_all])
info.index.name = 'mean_working'
print(info.to_string())

d = np.abs(iso_all - MEDIAN)
print(f'\n차단 레벨 최대 {d[d < 0.05].max():.4f}, 허용 레벨 최소 {d[d > 0.05].min():.4f} '
      f'(약 {d[d > 0.05].min() / d[d < 0.05].max():.0f}배)')

                 n    등위회귀  |중앙값과 차이|
mean_working                         
4                5  0.2780     0.2020
5               20  0.3055     0.1745
6               94  0.3148     0.1652
7              318  0.4657     0.0143
8              451  0.4700     0.0100
9              537  0.4700     0.0100
10             346  0.4700     0.0100
11             120  0.5964     0.1164
12              26  0.7236     0.2436
13              23  0.7236     0.2436
14               9  0.7236     0.2436
15              17  0.7236     0.2436
16               2  0.7236     0.2436
결측            1032  0.4821     0.0021

차단 레벨 최대 0.0143, 허용 레벨 최소 0.1164 (약 8배)


## 4. 게이트

두 조건이 모두 참일 때만 SVR 예측을 폴백으로 **교체**한다. 섞지 않는다.

In [4]:
CUT, THR = 0.026, 0.04

def oof(seed):
    ps = np.zeros(len(y)); fb = np.zeros(len(y))
    for t, v in KFold(5, shuffle=True, random_state=seed).split(X):
        ps[v] = np.clip(model().fit(X[t], y[t]).predict(X[v]), 0, 1)
        fb[v] = iso_table(LVL[t], y[t], LVL[v])
    return ps, fb

def blend32(ps, fb, tau=0.03, thr=THR):
    w = np.exp(-((np.abs(ps - MEDIAN) / tau) ** 2)) * (np.abs(fb - MEDIAN) > thr)
    return np.clip((1 - w) * ps + w * fb, 0, 1)

def switch33(ps, fb, cut=CUT, thr=THR):
    use = (np.abs(ps - MEDIAN) < cut) & (np.abs(fb - MEDIAN) > thr)
    return np.where(use, fb, ps)

## 5. 검증

시드 10개로 대응비교한다. 뒤의 4개는 32번에서도 쓰지 않은 시드다.
같은 시드·같은 폴드·같은 SVR 예측 위에서 결합 방식만 바꾼다.

In [5]:
rows = []
for s in SEEDS + NEW_SEEDS:
    ps, fb = oof(s)
    rows.append({'시드': s, '구분': '기존' if s in SEEDS else '신규',
                 'SVR 단독': mean_absolute_error(y, ps),
                 '32번 블렌드': mean_absolute_error(y, blend32(ps, fb)),
                 '33번 스위치': mean_absolute_error(y, switch33(ps, fb))})

res = pd.DataFrame(rows)
res['차이'] = res['33번 스위치'] - res['32번 블렌드']
print(res.round(6).to_string(index=False))
print(f"\n평균 차이 {res['차이'].mean():+.6f}, 부호 일관 {bool((res['차이'] < 0).all())}")
print(f"상수 {MEDIAN} 기준선 {mean_absolute_error(y, np.full_like(y, MEDIAN)):.6f}")

   시드 구분   SVR 단독  32번 블렌드  33번 스위치        차이
   42 기존 0.147668 0.145008 0.144894 -0.000114
 2024 기존 0.144505 0.141411 0.141302 -0.000109
    7 기존 0.146518 0.143736 0.143570 -0.000167
  123 기존 0.145962 0.143301 0.143124 -0.000176
  999 기존 0.145809 0.142705 0.142605 -0.000100
 2025 기존 0.149742 0.146294 0.146105 -0.000189
31337 신규 0.147193 0.144523 0.144405 -0.000118
    5 신규 0.146467 0.143557 0.143425 -0.000132
 2718 신규 0.148250 0.145340 0.145167 -0.000173
 1618 신규 0.145674 0.143062 0.142968 -0.000094

평균 차이 -0.000137, 부호 일관 True
상수 0.48 기준선 0.249443


시드 10개 전부에서 개선됐다.

28번의 GroupKFold(근접 중복행을 한 폴드로 묶어 중복 효과를 제거한 검증)로 그룹 순열을
바꿔가며 8번 재면 32번 0.245448, 33번 **0.245331** 로 개선된다. 상수 기준선은 0.249443 이다.
중복을 걷어내도 이득이 남는다. 해당 검증 코드는 쌍 탐지를 포함하므로 28번에 두고
제출 노트북에는 넣지 않았다.

## 6. c 와 THR 고원

두 값이 튜닝된 것이 아니라 넓은 구간에서 같은 결과를 내는지 본다.

In [6]:
ps, fb = oof(42)
print('cutoff c (THR=0.04 고정)')
print(f'{"c":>8}{"MAE":>12}{"교체 행":>9}')
for c in [0.010, 0.014, 0.018, 0.022, 0.026, 0.030, 0.034, 0.040, 0.050]:
    m = mean_absolute_error(y, switch33(ps, fb, cut=c))
    n = int(((np.abs(ps - MEDIAN) < c) & (np.abs(fb - MEDIAN) > THR)).sum())
    print(f'{c:>8.3f}{m:>12.6f}{n:>9}')

print('\nTHR (c=0.026 고정)')
print(f'{"THR":>8}{"MAE":>12}')
for thr in [0.02, 0.04, 0.06, 0.08, 0.10, 0.12]:
    print(f'{thr:>8.2f}{mean_absolute_error(y, switch33(ps, fb, thr=thr)):>12.6f}')

cutoff c (THR=0.04 고정)
       c         MAE     교체 행
   0.010    0.146568      107
   0.014    0.144841      195
   0.018    0.144841      195
   0.022    0.144943      199
   0.026    0.144894      200
   0.030    0.144987      201
   0.034    0.144962      202
   0.040    0.145131      206
   0.050    0.145261      209

THR (c=0.026 고정)
     THR         MAE
    0.02    0.144931
    0.04    0.144894
    0.06    0.144894
    0.08    0.144894
    0.10    0.144894
    0.12    0.145361


`c` 는 0.022~0.030, `THR` 은 0.04~0.08 에서 사실상 평평하다.
10시드·GroupKFold 양쪽을 모두 통과하는 구간이 `c` 0.022~0.030 이고 그 가운데를 썼다.
`THR` 은 레벨들이 두 무리로 8배 벌어져 있어 그 사이 어디를 잘라도 같은 분할이 나온다.

## 7. 최종 학습 및 제출

In [7]:
final_svr = model().fit(X, y)
pred_svr = np.clip(final_svr.predict(X_test), 0, 1)
pred_fb = iso_table(LVL, y, LVL_TEST)
pred = switch33(pred_svr, pred_fb)

use = (np.abs(pred_svr - MEDIAN) < CUT) & (np.abs(pred_fb - MEDIAN) > THR)
print(f'폴백으로 교체된 행 : {int(use.sum())}개 ({use.mean()*100:.1f}%)')
print(f'교체 행 평균 이동폭 : {np.abs(pred - pred_svr)[use].mean():.4f}')
print(f'32번 대비 예측 변화 : {np.abs(pred - blend32(pred_svr, pred_fb)).mean():.4f}')
print(f'예측 평균 {pred.mean():.4f}, 표준편차 {pred.std():.4f}, '
      f'범위 {pred.min():.3f}~{pred.max():.3f}')

sub = pd.read_csv('../data/sample_submission.csv')
sub['stress_score'] = pred
sub.to_csv('../submissions/submit_33_hard_switch.csv', index=False)
print('\nsaved -> submissions/submit_33_hard_switch.csv')
print(sub.head().to_string(index=False))

폴백으로 교체된 행 : 183개 (6.1%)
교체 행 평균 이동폭 : 0.1646
32번 대비 예측 변화 : 0.0010
예측 평균 0.5008, 표준편차 0.2040, 범위 0.000~1.000

saved -> submissions/submit_33_hard_switch.csv
       ID  stress_score
TEST_0000          0.49
TEST_0001          0.97
TEST_0002          0.19
TEST_0003          0.49
TEST_0004          0.53


## 8. 정리

| 항목 | 29번 | 32번 | 33번 |
|---|---|---|---|
| SVR | C=4.0 gamma=2.0 | 동일 | 동일 |
| 폴백 추정기 | 레벨 EB 평균 (k=20) | 등위회귀 | 등위회귀 |
| 폴백 조건 | 예측이 중앙값 근처 | + 폴백이 중앙값과 다를 것 | 동일 |
| 결합 방식 | 가우시안 블렌드 | 가우시안 블렌드 | **교체 (하드 스위치)** |
| 조정 값 | tau | tau, THR | c, THR |
| CV (시드 10개) | 0.144739 | 0.143894 | 0.143757 |
| GroupKFold (8분할) | 0.246673 | 0.245448 | 0.245331 |
| 리더보드 | 0.1272127 | 0.1262152 | — |

볼록결합은 제곱오차 기준의 최적이다. 이 대회 지표는 MAE 이고, 사후분포가 이봉형이면
두 봉우리 사이의 평균은 어느 쪽도 아닌 값이 된다. 섞지 않고 고르면 그 손해가 없어진다.

부수적으로 `tau` 와 `wmax` 가 사라진다. 조정할 값이 `c` 와 `THR` 둘뿐이고 둘 다 고원이다.

### 규정 관련

- 스케일러·인코더·등위회귀 전부 train(또는 학습 폴드)에서만 적합하고 test 에는 적용만 했다
- 근접 중복행 탐색, 최근접이웃 매칭, test 행 간 정보 공유 코드가 없다
- 게이트 두 조건 모두 모델 자기 출력과 train 레벨 통계만 쓴다